# Fase 9 — Lengan Resep: Mamba-YOLO-T (B2) dengan `optimizer="auto"`

**Pertanyaan:** apakah baseline kita tertinggal karena **resep**, bukan karena arsitektur?

Seluruh 50 run kita memakai `optimizer="SGD", lr0=0.002` — dipilih demi kestabilan varian
deformable, bukan karena wajar bagi B2. Satu run eksplorasi B2 seed 1 dgn setelan otomatis
pernah mencapai **0.675** (vs 0.5698 di resep terkunci). Notebook ini mengulanginya secara
terkendali.

### ⚠️ Koreksi istilah (diverifikasi dari kode)
Catatan lama menyebut run itu "lr0=0.01". **Itu keliru.** `trainer.py` menyatakan eksplisit bahwa
`optimizer="auto"` **mengabaikan `lr0` dan `momentum`**. Utk dataset kita:

    iterations = ceil(2726 / max(8, 64)) x 150 = 6450   (<= 10000)
    -> AdamW, lr = round(0.002*5/(4+8), 6) = 0.000833, momentum = 0.9, warmup_bias_lr = 0.0

Jadi LR sebenarnya **0.000833 — LEBIH RENDAH dari 0.002 milik kita.**
Perbedaannya bukan "LR terlalu rendah", melainkan **SGD vs AdamW**.

### Yang dikendalikan
Hanya optimizer yang berubah. 150 epoch, batch 8, 640 px, `patience=0`, `cos_lr=False`,
`deterministic=True`, TF32 **mati**, seed 1 — semuanya sama seperti 50 run kita.

⚠️ Run 0.675 dulu kemungkinan **tidak** mematikan TF32. Kita tetap mematikannya demi konsistensi;
kalau hasilnya meleset sedikit dari 0.675, TF32 adalah tersangka pertama.

### Cara membaca hasil
| Hasil | Artinya |
|---|---|
| ~0.57 (mirip resep terkunci) | Resep bukan penjelasannya → Tabel T2 aman, 2 run pembanding TIDAK perlu |
| ~0.67 (melonjak) | Baseline kita memang tertinggal → jalankan YOLOv8n & v8s di resep sama utk cek urutan |
| Di antaranya | Laporkan apa adanya; besarannya menentukan langkah berikutnya |

## 1. Mount Drive + Kredensial Kaggle

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, shutil, subprocess
DRIVE_ROOT   = "/content/drive/MyDrive/demamba-yolowd"
DRIVE_OUTPUT = f"{DRIVE_ROOT}/output"
os.makedirs(DRIVE_OUTPUT, exist_ok=True); os.makedirs(f"{DRIVE_ROOT}/analysis", exist_ok=True)

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy(f"{DRIVE_ROOT}/kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
print("Drive + Kaggle siap")

Mounted at /content/drive
Drive + Kaggle siap


## 2. Dataset + GPU

In [2]:
import torch
DATA_DIR = "/content/data"
if not glob.glob(f"{DATA_DIR}/**/*.txt", recursive=True):
    os.makedirs(DATA_DIR, exist_ok=True)
    subprocess.run(["kaggle","datasets","download","-d","benyxu/dsf-yolo-steel-tube-dataset",
                    "-p",DATA_DIR,"--unzip"], check=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA")
nw = min(os.cpu_count() // max(torch.cuda.device_count(), 1), 8)
print(f"cpu_count={os.cpu_count()} -> dataloader workers={nw}  (CATAT: harus sama dgn 50 run kita)")
assert torch.cuda.is_available(), "GPU tidak aktif -- Runtime > Change runtime type"

GPU: NVIDIA A100-SXM4-40GB
cpu_count=12 -> dataloader workers=8  (CATAT: harus sama dgn 50 run kita)


## 3. 🔧 Konfigurasi

In [3]:
SEED, TARGET_EPOCHS = 1, 150
OPTIMIZER   = "auto"      # <- SATU-SATUNYA yang berbeda dari 50 run kita (dulu: "SGD" + lr0=0.002)
USE_COS_LR  = False       # sama spt sebelumnya
MODEL_YAML  = "ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml"
MODEL_LABEL = "B2-auto"   # label BEDA -> tidak menimpa baris `B2` di CSV lama
RUN_NAME    = f"b2_seed{SEED}_{TARGET_EPOCHS}ep_auto"
EXPECTED_PARAMS = 5_986_912      # Mamba-YOLO-T @ nc=8

print(f"MODEL    : Mamba-YOLO-T  (label CSV: {MODEL_LABEL})")
print(f"RUN_NAME : {RUN_NAME}")
print(f"optimizer: {OPTIMIZER}  -> framework akan memilih AdamW lr=0.000833 (lihat log 'optimizer:')")
print(f"pembanding resep terkunci: B2 seed1 = 0.5698 | run eksplorasi lama = 0.675")

MODEL    : Mamba-YOLO-T  (label CSV: B2-auto)
RUN_NAME : b2_seed1_150ep_auto
optimizer: auto  -> framework akan memilih AdamW lr=0.000833 (lihat log 'optimizer:')
pembanding resep terkunci: B2 seed1 = 0.5698 | run eksplorasi lama = 0.675


## 4. Clone Mamba-YOLO + Patch `torch.load` + Build Kernel

B2 adalah Mamba-YOLO-T asli, jadi **tidak ada patch deformable** — hanya patch `torch.load`.
Kernel `selective_scan` tetap wajib (B2 memang memakainya). Wheel di-cache di Drive.

In [4]:
import re, py_compile
REPO = "/content/Mamba-YOLO"
os.chdir("/content")
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(["git","clone","--depth","1","https://github.com/HZAI-ZJNU/Mamba-YOLO.git",REPO], check=True)

def patch(path, subs):
    p = f"{REPO}/{path}"; src = open(p).read()
    for old, new, tag in subs:
        if new in src: print(f"  ~ {tag}: skip"); continue
        assert src.count(old) == 1, f"GAGAL {tag}: {src.count(old)}x (harus 1x)"
        src = src.replace(old, new); print(f"  OK {tag}")
    open(p, "w").write(src); py_compile.compile(p, doraise=True)

print("### Patch torch.load")
patch("ultralytics/nn/tasks.py", [
    ('            ckpt = torch.load(file, map_location="cpu")\n\n    except ModuleNotFoundError as e:',
     '            ckpt = torch.load(file, map_location="cpu", weights_only=False)\n\n    except ModuleNotFoundError as e:', "4.1"),
    ('        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu")',
     '        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu", weights_only=False)', "4.2"),
])
patch("ultralytics/utils/torch_utils.py", [
    ('    x = torch.load(f, map_location=torch.device("cpu"))\n    if "model" not in x:',
     '    x = torch.load(f, map_location=torch.device("cpu"), weights_only=False)\n    if "model" not in x:', "5.1"),
])

subprocess.run([sys.executable,"-m","pip","install","-q","ninja"], check=True)
os.environ["MAX_JOBS"] = str(max(1, int(subprocess.run(["nproc"],capture_output=True,text=True).stdout.strip() or "2")))
cap = torch.cuda.get_device_capability(0)
SS_TAG = f"sm{cap[0]}{cap[1]}_torch{torch.__version__.split('+')[0]}_py{sys.version_info.major}{sys.version_info.minor}"
SS_WHEEL_DIR = f"{DRIVE_ROOT}/wheels/selective_scan_{SS_TAG}"; os.makedirs(SS_WHEEL_DIR, exist_ok=True)
cached = glob.glob(f"{SS_WHEEL_DIR}/*.whl")
if cached:
    print(f"Wheel cache ({SS_TAG}) -- install langsung")
    subprocess.run([sys.executable,"-m","pip","install"]+cached, check=True)
else:
    print(f"Belum ada cache {SS_TAG} -- compile sekali, lalu disimpan ke Drive")
    subprocess.run([sys.executable,"-m","pip","wheel",".","--no-deps","-v","-w",SS_WHEEL_DIR],
                   cwd=f"{REPO}/selective_scan", check=True)
    subprocess.run([sys.executable,"-m","pip","install"]+glob.glob(f"{SS_WHEEL_DIR}/*.whl"), check=True)

subprocess.run([sys.executable,"-m","pip","install","-q","-e","."], cwd=REPO, check=True)
for k in [x for x in list(sys.modules) if x.startswith("ultralytics")]: del sys.modules[k]
import selective_scan_cuda_core; print("selective_scan_cuda_core OK")
subprocess.run([sys.executable,"-m","pip","install","-q","seaborn","thop","timm","einops"], check=True)
subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","wandb","ray","tensorboard","tensorboardX",
                "tensorboard-data-server","clearml","comet_ml","dvclive","mlflow"], check=False)
print("Lingkungan siap")

### Patch torch.load
  OK 4.1
  OK 4.2
  OK 5.1
Wheel cache (sm80_torch2.11.0_py313) -- install langsung
selective_scan_cuda_core OK
Lingkungan siap


## 5. data.yaml — sama persis dgn 40 run DSF sebelumnya

In [5]:
import yaml
from pathlib import Path
cands = [p.parent.parent for p in Path(DATA_DIR).rglob("images/train2021") if p.is_dir()]
assert cands, f"struktur images/train2021 tidak ditemukan di {DATA_DIR}"
DSF_BASE = str(cands[0].resolve())
NAMES = ["air-hole","bite-edge","broken-arc","crack","dense-gas-hole","non-fusion","slag-inclusion","unfused-hole"]
DATA_YAML = f"{REPO}/ultralytics/cfg/datasets/dsf-yolo.yaml"
yaml.safe_dump({"path":DSF_BASE,"train":"images/train2021","val":"images/val2021",
                "nc":len(NAMES),"names":NAMES}, open(DATA_YAML,"w"), sort_keys=False)
for s in ["train2021","val2021"]:
    d = Path(DSF_BASE)/"images"/s
    print(f"  {s}: {len([f for f in d.iterdir() if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'}])} citra")
assert (Path(DSF_BASE)/"labels"/"train2021").is_dir(), "folder labels/ tidak ada"
print(f"\nDSF_BASE: {DSF_BASE}\nkelas   : {NAMES}")

  train2021: 2726 citra
  val2021: 682 citra

DSF_BASE: /content/data/steel-tube-dataset-all/yolo
kelas   : ['air-hole', 'bite-edge', 'broken-arc', 'crack', 'dense-gas-hole', 'non-fusion', 'slag-inclusion', 'unfused-hole']


## 6. Pre-flight — pastikan arsitekturnya persis B2

In [6]:
os.chdir(REPO)
from copy import deepcopy
from ultralytics import YOLO
from ultralytics.nn.tasks import parse_model, yaml_model_load
d = yaml_model_load(MODEL_YAML); d["nc"] = len(NAMES)
m,_ = parse_model(deepcopy(d), ch=3, verbose=False)
n = sum(p.numel() for p in m.parameters()); del m
print(f"Mamba-YOLO-T: {n:,} parameter (harapan {EXPECTED_PARAMS:,})")
assert n == EXPECTED_PARAMS, "arsitektur BUKAN B2 yang sama -- hentikan."
print("Pre-flight OK -- arsitektur identik dgn 5 run B2 sebelumnya")

WARNING ⚠️ no model scale passed. Assuming scale='T'.


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/Mamba-YOLO/ultralytics/nn/modules/common_utils_mbyolo.py:103: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/content/Mamba-YOLO/ultralytics/nn/modules/common_utils_mbyolo.py:130: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd


Mamba-YOLO-T: 5,986,912 parameter (harapan 5,986,912)
Pre-flight OK -- arsitektur identik dgn 5 run B2 sebelumnya


## 7. Status checkpoint

In [7]:
last_ckpt = Path(DRIVE_OUTPUT)/RUN_NAME/"weights"/"last.pt"
if not last_ckpt.exists():
    STATUS = "BARU"; print("STATUS: BARU -- training mulai dari nol.")
else:
    raw = torch.load(str(last_ckpt), map_location="cpu", weights_only=False)
    ep, has_opt = raw.get("epoch",-1), raw.get("optimizer") is not None
    STATUS = "BELUM_SELESAI" if (has_opt and 0 <= ep < TARGET_EPOCHS-1) else "SELESAI"
    print(f"STATUS: {STATUS} (epoch={ep}, optimizer={has_opt})")

STATUS: BARU -- training mulai dari nol.


## 8. Training

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

if STATUS == "BARU":
    model = YOLO(MODEL_YAML)
    model.train(
        data=DATA_YAML, epochs=TARGET_EPOCHS,
        batch=8, imgsz=640, device=0, amp=True, seed=SEED,
        patience=0, cache=True, deterministic=True, workers=8,
        optimizer=OPTIMIZER,   # "auto" -> framework memilih; lr0 & momentum DIABAIKAN
        cos_lr=USE_COS_LR,
        project=DRIVE_OUTPUT, name=RUN_NAME, plots=False,
    )
elif STATUS == "BELUM_SELESAI":
    model = YOLO(str(last_ckpt)); model.train(resume=True)
else:
    model = YOLO(str(last_ckpt)); print("Sudah selesai -- tidak training ulang.")

SAVE_DIR = model.trainer.save_dir if getattr(model,"trainer",None) else Path(DRIVE_OUTPUT)/RUN_NAME
print(f"\nSAVE_DIR: {SAVE_DIR}")
print("PENTING: cari baris 'optimizer:' di log di atas & CATAT optimizer + lr yang dipilih framework.")
fin = torch.load(str(Path(SAVE_DIR)/"weights"/"last.pt"), map_location="cpu", weights_only=False)
print("final_eval selesai" if fin.get("optimizer") is None else f"BELUM tuntas (epoch={fin.get('epoch')}) -- jalankan ulang utk resume")

WARNING ⚠️ no model scale passed. Assuming scale='T'.
New https://pypi.org/project/ultralytics/8.4.164 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: task=detect, mode=train, model=ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml, data=/content/Mamba-YOLO/ultralytics/cfg/datasets/dsf-yolo.yaml, epochs=150, time=None, patience=0, batch=8, imgsz=640, save=True, save_period=-1, cache=True, device=0, workers=8, project=/content/drive/MyDrive/demamba-yolowd/output, name=b2_seed1_150ep_auto, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=1, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False

100%|██████████| 755k/755k [00:00<00:00, 121MB/s]

Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.

                   from  n    params  module                                       arguments                     
  0                  -1  1      5136  ultralytics.nn.modules.mamba_yolo.SimpleStem [3, 32, 3]                    
  1                  -1  1     33692  ultralytics.nn.modules.mamba_yolo.VSSBlock   [32, 32]                      
  2                  -1  1      8384  ultralytics.nn.modules.mamba_yolo.VisionClueMerge[32, 64]                      
  3                  -1  1    104184  ultralytics.nn.modules.mamba_yolo.VSSBlock   [64, 64]                      
  4                  -1  1     33152  ultralytics.nn.modules.mamba_yolo.VisionClueMerge[64, 128]                     
  5                  -1  3   1067892  ultralytics.nn.modules.mamba_yolo.VSSBlock   [128, 128]                    
  6                  -1  1    131840  ultralytics.nn.modules.mamba_yolo.VisionClueMerge[128, 256]    

Mamba-YOLO-T summary: 404 layers, 5986912 parameters, 5986896 gradients, 13.6 GFLOPs

Freezing layer 'model.21.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks with YOLOv8n...


/content/Mamba-YOLO/ultralytics/utils/checks.py:651: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(True):


...

  0%|          | 0/341 [00:00<?, ?it/s]/content/Mamba-YOLO/ultralytics/engine/trainer.py:374: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self.amp):
    150/150      5.72G      1.162     0.6383      1.063          6        640: 100%|██████████| 341/341 [00:56<00:00,  6.08it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 43/43 [00:04<00:00,  9.20it/s]


                   all        682       1381      0.957      0.949      0.971      0.682

150 epochs completed in 2.607 hours.
Optimizer stripped from /content/drive/MyDrive/demamba-yolowd/output/b2_seed1_150ep_auto/weights/last.pt, 12.3MB
Optimizer stripped from /content/drive/MyDrive/demamba-yolowd/output/b2_seed1_150ep_auto/weights/best.pt, 12.3MB

Validating /content/drive/MyDrive/demamba-yolowd/output/b2_seed1_150ep_auto/weights/best.pt...
Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Mamba-YOLO-T summary: 388 layers, 5985568 parameters, 0 gradients, 13.6 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 43/43 [00:04<00:00, 10.26it/s]

                   all        682       1381      0.957      0.949      0.971      0.682
              air-hole        406       1070      0.891      0.822      0.928      0.436
             bite-edge          8          8       0.92          1      0.995      0.749
            broken-arc         74         74      0.935      0.969      0.985      0.846
                 crack         23         23      0.987      0.826      0.906      0.549
        dense-gas-hole         43         43          1      0.998      0.995      0.793
            non-fusion         40         41      0.971      0.976       0.98      0.762
        slag-inclusion         27         27      0.983          1      0.995      0.625
          unfused-hole         95         95      0.968          1      0.985      0.696
Speed: 0.1ms preprocess, 3.5ms inference, 0.0ms loss, 0.8ms postprocess per image



SAVE_DIR: /content/drive/MyDrive/demamba-yolowd/output/b2_seed1_150ep_auto
PENTING: cari baris 'optimizer:' di log di atas & CATAT optimizer + lr yang dipilih framework.
final_eval selesai


## 9. Evaluasi + CSV terpisah

In [ ]:
import pandas as pd
OUT_CSV = f"{DRIVE_ROOT}/analysis/per_class_results_recipe.csv"
best = Path(SAVE_DIR)/"weights"/"best.pt"
assert best.exists(), "best.pt belum ada"
r = YOLO(str(best)).val(data=DATA_YAML, split="val", verbose=False)
base = dict(model=MODEL_LABEL, seed=SEED, ckpt_path=str(best))
rows = [{**base, "class":"ALL", "mAP50_95":r.box.map, "mAP50":r.box.map50,
         "precision":r.box.mp, "recall":r.box.mr}]
for i,c in enumerate(NAMES):
    rows.append({**base, "class":c, "mAP50_95":float(r.box.maps[i]),
                 "mAP50":float("nan"), "precision":float("nan"), "recall":float("nan")})
df = pd.DataFrame(rows)
if os.path.exists(OUT_CSV):
    df = pd.concat([pd.read_csv(OUT_CSV), df]).drop_duplicates(["model","seed","class"], keep="last")
df.to_csv(OUT_CSV, index=False); shutil.copy(OUT_CSV, "/content/per_class_results_recipe.csv")
v = df[(df["class"]=="ALL") & (df.model==MODEL_LABEL)].mAP50_95.iloc[0]
print(df[df["class"]=="ALL"][["model","seed","mAP50_95","mAP50","precision","recall"]].to_string(index=False))
print(f"\n  resep terkunci (SGD lr0=0.002), B2 seed1 : 0.5698")
print(f"  run eksplorasi lama (auto)              : ~0.675")
print(f"  RUN INI                                 : {v:.4f}")
print(f"\nTersimpan: {OUT_CSV} -- kirim `per_class_results_recipe.csv` ke sesi analisis.")

Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Mamba-YOLO-T summary: 388 layers, 5985568 parameters, 0 gradients, 13.6 GFLOPs


val: Scanning /content/data/steel-tube-dataset-all/yolo/labels/val2021.cache... 682 images, 2 backgrounds, 0 corrupt: 100%|██████████| 682/682 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 43/43 [00:06<00:00,  6.29it/s]


                   all        682       1381      0.951      0.948      0.971       0.68
Speed: 0.1ms preprocess, 5.3ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /content/Mamba-YOLO/runs/detect/val
  model  seed  mAP50_95    mAP50  precision   recall
B2-auto     1  0.679606 0.970531   0.951208 0.947711

  resep terkunci (SGD lr0=0.002), B2 seed1 : 0.5698
  run eksplorasi lama (auto)              : ~0.675
  RUN INI                                 : 0.6796

Tersimpan: /content/drive/MyDrive/demamba-yolowd/analysis/per_class_results_recipe.csv -- kirim `per_class_results_recipe.csv` ke sesi analisis.


## 10. Matikan Runtime (Jalankan Terakhir)

In [ ]:
print("Semua proses selesai. Runtime akan di-disconnect otomatis...")
import time
time.sleep(10)
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f"Gagal disconnect otomatis ({e})")
    print("Matikan manual lewat: Runtime > Disconnect and delete runtime")